# MAPF Algorithm Comparison: Cooperative A*, POPS, and PBS

This notebook compares the three methods on **the exact same grid, starts, and goals**.

- **Cooperative A\***: fixed input priority order.
- **POPS**: tests a bounded set of priority orders and keeps its best result.
- **PBS**: branches on pairwise priority constraints when conflicts occur.

> Keep `cooperative_astar.py`, `pops_cooperative_astar.py`, and `pbs_mapf.py` in the same folder as this notebook. Input CSVs should be in `results/`: `grid.csv`, `start.csv`, and `goal.csv`. Grid values: 0 = free, 1 = obstacle.

In [ ]:
# Install once if needed:
# %pip install numpy pandas matplotlib

import os
import time
import numpy as np
import pandas as pd
from IPython.display import display

from cooperative_astar import prioritized_cooperative_astar
from pops_cooperative_astar import prioritized_order_portfolio_search
from pbs_mapf import priority_based_search

# Edit this if your CSV files are stored elsewhere.
DATA_FOLDER = "results"
MAX_ORDERS = 120       # POPS: tested priority orders
SEED = 42              # reproducible POPS sampling
PBS_NODE_LIMIT = 1000  # increase if PBS stops before finding a solution
MAX_TIME = None        # None uses the algorithm's default time horizon
OUTPUT_CSV = os.path.join(DATA_FOLDER, "comparison_results.csv")

In [ ]:
def load_inputs(folder=DATA_FOLDER):
    required = ["grid.csv", "start.csv", "goal.csv"]
    missing = [name for name in required if not os.path.isfile(os.path.join(folder, name))]
    if missing:
        raise FileNotFoundError(
            f"Missing {missing} in {folder!r}. Generate/save grid.csv, start.csv, and goal.csv first."
        )
    grid = np.loadtxt(os.path.join(folder, "grid.csv"), delimiter=",", dtype=int)
    starts_arr = np.atleast_2d(np.loadtxt(os.path.join(folder, "start.csv"), delimiter=",", dtype=int))
    goals_arr = np.atleast_2d(np.loadtxt(os.path.join(folder, "goal.csv"), delimiter=",", dtype=int))
    starts = [tuple(map(int, row)) for row in starts_arr]
    goals = [tuple(map(int, row)) for row in goals_arr]
    if len(starts) != len(goals):
        raise ValueError("Number of starts and goals differ.")
    if len(set(starts)) != len(starts) or len(set(goals)) != len(goals):
        raise ValueError("Starts and goals should be unique.")
    return grid, starts, goals


grid, starts, goals = load_inputs()
print(f"Grid: {grid.shape[0]} x {grid.shape[1]}")
print(f"Robots: {len(starts)} | Obstacles: {int(np.sum(grid == 1))}")
print("Starts:", starts)
print("Goals: ", goals)
print("Input validation complete.")

## Run the three algorithms

Makespan and total path length are reported only when **all robots succeed**. Partial solutions are not comparable as complete schedules.

In [ ]:
def summarise(name, results, n, elapsed, notes=""):
    if results is None:
        return {
            "Algorithm": name, "Solved robots": f"0/{n}", "Success rate (%)": 0.0,
            "Makespan": np.nan, "Total path length": np.nan,
            "Nodes expanded": np.nan, "Runtime (s)": round(elapsed, 6), "Notes": notes
        }
    robots = [results[f"Robot {i+1}"] for i in range(n)]
    count = sum(bool(r.get("success")) for r in robots)
    complete = count == n
    arrivals = [r.get("arrival_time") for r in robots if r.get("success")]
    lengths = [r.get("path_length") for r in robots if r.get("success")]
    nodes = sum(r.get("nodes_expanded", 0) for r in robots)
    return {
        "Algorithm": name, "Solved robots": f"{count}/{n}",
        "Success rate (%)": round(100 * count / n, 2),
        "Makespan": max(arrivals) if complete and arrivals else np.nan,
        "Total path length": sum(lengths) if complete else np.nan,
        "Nodes expanded": nodes, "Runtime (s)": round(elapsed, 6), "Notes": notes
    }

n = len(starts)
rows = []

# 1) Fixed-priority Cooperative A*
t0 = time.perf_counter()
coop_results = prioritized_cooperative_astar(grid, starts, goals, max_time=MAX_TIME)
coop_seconds = time.perf_counter() - t0
rows.append(summarise("Fixed-priority Cooperative A*", coop_results, n, coop_seconds,
                      "Input order is used as priority"))

# 2) POPS: bounded priority-order portfolio search
t0 = time.perf_counter()
pops_output = prioritized_order_portfolio_search(
    grid, starts, goals, max_orders=MAX_ORDERS, seed=SEED, max_time=MAX_TIME
)
pops_seconds = time.perf_counter() - t0
rows.append(summarise("POPS", pops_output["results"], n, pops_seconds,
                      f"Orders tested: {pops_output['orders_tested']}; best order: {pops_output['best_priority_order']}"))

# 3) PBS: priority constraint search
t0 = time.perf_counter()
pbs_output = priority_based_search(
    grid, starts, goals, max_time=MAX_TIME, max_high_level_nodes=PBS_NODE_LIMIT
)
pbs_seconds = time.perf_counter() - t0
rows.append(summarise("PBS", pbs_output.get("results"), n, pbs_seconds,
                      f"High-level nodes: {pbs_output.get('high_level_nodes')}; {pbs_output.get('reason')}"))

comparison = pd.DataFrame(rows)
display(comparison)

os.makedirs(DATA_FOLDER, exist_ok=True)
comparison.to_csv(OUTPUT_CSV, index=False)
print(f"Saved comparison table to: {OUTPUT_CSV}")
print("PBS status:", pbs_output.get("reason"))
if pbs_output.get("success"):
    print("PBS priority constraints (higher-priority robot, lower-priority robot):",
          pbs_output.get("priority_constraints"))

## Interpret the result carefully

- Compare makespan only when all robots have valid paths.
- POPS is only as good as the priority orders it tests; 120 orders are a sample, not an exhaustive search for 12 robots.
- PBS may stop at `PBS_NODE_LIMIT`; a failed run can mean the limit/horizon was insufficient, not necessarily that no solution exists.
- This educational PBS implementation prioritizes candidate schedules by makespan and path length, but it does **not** prove globally optimal makespan.
- Run multiple maps/seeds and report average success rate, runtime, and makespan for successful instances. Do not present a single grid as conclusive evidence.